# Mạng Residual (Residual Networks)

Chào mừng bạn đến với bài tập thứ hai của tuần này! Bạn sẽ học cách xây dựng các mạng tích chập (CNN) rất sâu bằng cách sử dụng Mạng Residual (ResNet). Về mặt lý thuyết, các mạng rất sâu có khả năng biểu diễn những hàm số cực kỳ phức tạp; tuy nhiên, trên thực tế, chúng lại rất khó huấn luyện. Mạng Residual, được giới thiệu bởi [He và cộng sự](https://arxiv.org/pdf/1512.03385.pdf), cho phép huấn luyện các mạng có độ sâu lớn hơn nhiều so với những gì có thể thực hiện được trước đây.

**Trong bài tập này, bạn sẽ:**
- Triển khai các khối xây dựng cơ bản của ResNet.
- Kết hợp các khối này để xây dựng và huấn luyện một mạng thần kinh tiên tiến (state-of-the-art) phục vụ bài toán phân loại hình ảnh.

## <font color='darkblue'>Các cập nhật</font>

#### Nếu bạn đang làm việc trên notebook trước khi có bản cập nhật này...
* Notebook hiện tại là phiên bản "2a".
* Bạn có thể tìm thấy phần công việc ban đầu của mình đã được lưu trong notebook với tên phiên bản cũ ("v2").
* Để xem danh sách tệp, hãy vào menu "File->Open"; thao tác này sẽ mở một thẻ (tab) mới hiển thị danh sách các tệp.

#### Danh sách các cập nhật
* Đối với việc kiểm thử trên ảnh, đã thay thế `preprocess_input(x)` bằng `x=x/255.0` để chuẩn hóa ảnh đầu vào theo cùng cách mà dữ liệu huấn luyện của mô hình đã được chuẩn hóa.
* Sử dụng thuật ngữ các lớp "nông hơn" (shallower) để chỉ những lớp gần với đầu vào, và các lớp "sâu hơn" (deeper) để chỉ những lớp gần với đầu ra (thay vì dùng từ "thấp hơn" hay "sớm hơn").
* Đã thêm/cập nhật các hướng dẫn.

This assignment will be done in Keras. 

Before jumping into the problem, let's run the cell below to load the required packages.

In [4]:
!pip install pydot
import numpy as np
import tensorflow as tf
import keras

from keras import layers
from keras.layers import (
    Input, Add, Dense, Activation, ZeroPadding2D,
    BatchNormalization, Flatten, Conv2D,
    AveragePooling2D, MaxPooling2D, GlobalMaxPooling2D
)

from keras.models import Model, load_model
from keras.preprocessing import image
from keras.utils import get_file, model_to_dot, plot_model
from keras.applications.resnet50 import preprocess_input
from keras.initializers import glorot_uniform

import pydot
from IPython.display import SVG
from PIL import Image
from matplotlib.pyplot import imshow

import keras.backend as K

# Cấu hình định dạng ảnh
keras.config.set_image_data_format('channels_last')

# Thư viện hỗ trợ bài ResNet
from resnets_utils import *

%matplotlib inline

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
TensorFlow: 2.21.0
Keras: 3.15.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1 - Vấn đề của các mạng nơ-ron rất sâu

Tuần trước, bạn đã xây dựng mạng nơ-ron tích chập (CNN) đầu tiên của mình. Trong những năm gần đây, các mạng nơ-ron ngày càng trở nên sâu hơn; các mạng tiên tiến nhất đã phát triển từ chỗ chỉ có vài lớp (ví dụ: AlexNet) lên đến hơn một trăm lớp.

*   Lợi ích chính của một mạng rất sâu là khả năng biểu diễn các hàm số cực kỳ phức tạp. Nó cũng có thể học các đặc trưng ở nhiều cấp độ trừu tượng khác nhau, từ các đường biên (ở những lớp nông hơn, gần với đầu vào) cho đến những đặc trưng rất phức tạp (ở những lớp sâu hơn, gần với đầu ra).
*   Tuy nhiên, việc sử dụng mạng sâu hơn không phải lúc nào cũng mang lại hiệu quả. Một rào cản lớn trong quá trình huấn luyện chúng là hiện tượng triệt tiêu gradient (vanishing gradients): ở các mạng rất sâu, tín hiệu gradient thường giảm nhanh về mức 0, khiến cho thuật toán hạ gradient (gradient descent) trở nên cực kỳ chậm chạp.
*   Cụ thể hơn, trong quá trình hạ gradient, khi thực hiện lan truyền ngược (backpropagation) từ lớp cuối cùng về lớp đầu tiên, bạn phải nhân với ma trận trọng số tại mỗi bước; do đó, giá trị gradient có thể giảm theo cấp số nhân về 0 (hoặc trong những trường hợp hiếm gặp, nó có thể tăng theo cấp số nhân và "bùng nổ" - exploding gradient - để đạt các giá trị rất lớn).
*   Vì vậy, trong quá trình huấn luyện, bạn có thể thấy độ lớn (hay chuẩn - norm) của gradient tại các lớp nông giảm rất nhanh về 0 khi quá trình huấn luyện diễn ra:

<img src="images/vanishing_grad_kiank.png" style="width:450px;height:220px;">
<caption><center> <u> <font color='purple'> **Figure 1** </u><font color='purple'>  : **Vanishing gradient** <br> The speed of learning decreases very rapidly for the shallower layers as the network trains </center></caption>

You are now going to solve this problem by building a Residual Network!

## 2 - Building a Residual Network

In ResNets, a "shortcut" or a "skip connection" allows the model to skip layers:  

<img src="images/skip_connection_kiank.png" style="width:650px;height:200px;">
<caption><center> <u> <font color='purple'> **Figure 2** </u><font color='purple'>  : A ResNet block showing a **skip-connection** <br> </center></caption>



Hình bên trái hiển thị "đường dẫn chính" đi qua mạng. Hình bên phải bổ sung một đường tắt vào đường dẫn chính đó. Bằng cách xếp chồng các khối ResNet này lên nhau, bạn có thể tạo ra một mạng rất sâu.

Trong bài giảng, chúng ta cũng đã thấy rằng việc sử dụng các khối ResNet có đường tắt giúp một khối bất kỳ dễ dàng học được hàm đồng nhất (identity function). Điều này có nghĩa là bạn có thể xếp chồng thêm các khối ResNet mà ít có nguy cơ làm giảm hiệu suất trên tập huấn luyện.

(Ngoài ra còn có bằng chứng cho thấy khả năng dễ dàng học hàm đồng nhất đóng góp vào hiệu suất vượt trội của ResNet nhiều hơn cả vai trò của kết nối bỏ qua trong việc khắc phục vấn đề triệt tiêu gradient).

Có hai loại khối chính được sử dụng trong ResNet, tùy thuộc chủ yếu vào việc kích thước đầu vào và đầu ra có giống nhau hay khác nhau. Bạn sẽ triển khai cả hai loại này: "khối đồng nhất" (identity block) và "khối tích chập" (convolutional block).

### 2.1 - Khối identity (Identity block)

Khối identity là khối tiêu chuẩn được sử dụng trong các mạng ResNet, tương ứng với trường hợp mà giá trị kích hoạt đầu vào (ví dụ: $a^{[l]}$) có cùng kích thước với giá trị kích hoạt đầu ra (ví dụ: $a^{[l+2]}$). Để làm rõ các bước diễn ra bên trong khối identity của ResNet, dưới đây là một sơ đồ khác minh họa chi tiết từng bước:

<img src="images/idblock2_kiank.png" style="width:650px;height:150px;">
<caption><center> <u> <font color='purple'> **Hình 3** </u><font color='purple'>  : **Khối identity.** Kết nối tắt (skip connection) "bỏ qua" 2 lớp. </center></caption>

Đường dẫn phía trên là "đường tắt" (shortcut path). Đường dẫn phía dưới là "đường dẫn chính" (main path). Trong sơ đồ này, chúng tôi cũng thể hiện rõ các bước CONV2D và ReLU ở mỗi lớp. Để đẩy nhanh quá trình huấn luyện, chúng tôi cũng đã thêm bước BatchNorm. Đừng lo lắng về việc triển khai nó sẽ phức tạp—bạn sẽ thấy rằng BatchNorm chỉ tốn một dòng mã trong Keras!

Trong bài tập này, bạn sẽ thực sự triển khai một phiên bản mạnh mẽ hơn một chút của khối identity này, trong đó kết nối tắt sẽ "bỏ qua" 3 lớp ẩn thay vì 2 lớp. Nó có dạng như sau:

<img src="images/idblock3_kiank.png" style="width:650px;height:150px;">
<caption><center> <u> <font color='purple'> **Hình 4** </u><font color='purple'>  : **Khối identity.** Kết nối tắt "bỏ qua" 3 lớp.</center></caption>

Dưới đây là các bước cụ thể.

Thành phần đầu tiên của nhánh chính:
- Lớp CONV2D thứ nhất có $F_1$ bộ lọc (filter) với kích thước (1,1) và bước trượt (stride) là (1,1). Chế độ padding là "valid" và tên lớp là `conv_name_base + '2a'`. Sử dụng giá trị 0 làm seed cho quá trình khởi tạo ngẫu nhiên.
- Lớp BatchNorm thứ nhất thực hiện chuẩn hóa theo trục 'channels' (kênh). Tên lớp là `bn_name_base + '2a'`.
- Sau đó áp dụng hàm kích hoạt ReLU. Lớp này không có tên và không có tham số siêu cấp (hyperparameter).

Thành phần thứ hai của nhánh chính:
- Lớp CONV2D thứ hai có $F_2$ bộ lọc với kích thước $(f,f)$ và bước trượt là (1,1). Chế độ padding là "same" và tên lớp là `conv_name_base + '2b'`. Sử dụng giá trị 0 làm seed cho quá trình khởi tạo ngẫu nhiên.
- Lớp BatchNorm thứ hai thực hiện chuẩn hóa theo trục 'channels'. Tên lớp là `bn_name_base + '2b'`.
- Sau đó áp dụng hàm kích hoạt ReLU. Lớp này không có tên và không có tham số siêu cấp.

Thành phần thứ ba của nhánh chính:
- Lớp CONV2D thứ ba có $F_3$ bộ lọc với kích thước (1,1) và bước trượt là (1,1). Chế độ padding là "valid" và tên lớp là `conv_name_base + '2c'`. Sử dụng giá trị 0 làm seed cho quá trình khởi tạo ngẫu nhiên.
- Lớp BatchNorm thứ ba thực hiện chuẩn hóa theo trục 'channels'. Tên lớp là `bn_name_base + '2c'`.
- Lưu ý rằng **không** có hàm kích hoạt ReLU trong thành phần này.

Bước cuối cùng:
- Cộng `X_shortcut` và đầu ra từ lớp thứ ba `X` lại với nhau.
- **Gợi ý**: Cú pháp sẽ có dạng `Add()([var1,var2])`
- Sau đó áp dụng hàm kích hoạt ReLU. Lớp này không có tên và không có tham số siêu cấp. **Bài tập**: Triển khai khối identity (identity block) của ResNet. Chúng tôi đã triển khai thành phần đầu tiên của nhánh chính (main path). Hãy đọc kỹ phần này để đảm bảo bạn hiểu rõ cơ chế hoạt động của nó. Bạn cần triển khai các phần còn lại.
- Để triển khai bước Conv2D: [Conv2D](https://keras.io/layers/convolutional/#conv2d)
- Để triển khai BatchNorm: [BatchNormalization](https://faroit.github.io/keras-docs/1.2.2/layers/normalization/) (tham số `axis`: Số nguyên, xác định trục cần chuẩn hóa - thường là trục 'channels')
- Đối với hàm kích hoạt (activation), hãy sử dụng: `Activation('relu')(X)`
- Để cộng giá trị được truyền qua nhánh tắt (shortcut): [Add](https://keras.io/layers/merge/#add)